In [2]:
!pip install biopython


   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 23.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import math

# Read counts matrix 
def read_counts_matrix(filename):
    counts = {'a': [], 'c': [], 'g': [], 't': []} # Initialize an empty dictionary to store counts for each base
    with open(filename, 'r') as f:
        for line in f:
            if '|' in line:
                base, data = line.strip().split('|') # Split the line into base (A/C/G/T) and its counts
                base = base.strip().lower()
                counts[base] = list(map(int, data.strip().split())) #Convert the space-separated string of numbers into a list of integers
    return counts

# Compute raw frequency matrix F(b, j) 
def compute_frequency_matrix(counts, pseudocount=False):
    freq_matrix = {b: [] for b in 'acgt'} #Initialize an empty frequency matrix with one list per base (A, C, G, T)
    num_positions = len(counts['a'])
    # Loop over each position in the motif
    for j in range(num_positions):
        total = sum(counts[b][j] + (1 if pseudocount else 0) for b in 'acgt') # Loop over each position in the motif
        # Compute the frequency for each base at position j
        for b in 'acgt':
            count = counts[b][j] + (1 if pseudocount else 0) 
            freq_matrix[b].append(count / total)
    return freq_matrix

# Compute log-odds weight matrix W(b, j)
def compute_log_odds_matrix(freq_matrix, background=0.25):
    log_odds = {b: [] for b in 'acgt'} # Initialize an empty log-odds matrix for each base (A, C, G, T)
    for b in 'acgt':
        # For each frequency value at position j, compute the log-odds score
        for f in freq_matrix[b]:
            log_odds[b].append(round(math.log2(f / background), 4))
    return log_odds

# Display matrix
def print_matrix(matrix, title):
    print(f"\n {title} ")
    for b in 'acgt':
         # Format each row to 4 decimal places and print
        print(f"{b.upper()}: {['{:.4f}'.format(x) for x in matrix[b]]}")

# MAIN 
if __name__ == "__main__":
    counts_file = "argR-counts-matrix.txt"  # Input file containing base counts
# Read the counts matrix from the input file
    counts = read_counts_matrix(counts_file)
    
    # Raw frequency matrix F(b, j)
    raw_freq = compute_frequency_matrix(counts, pseudocount=False)
    print_matrix(raw_freq, "F(b, j) - Raw Frequency Matrix")

    # Pseudocount-adjusted frequency matrix F′(b, j)
    pseudo_freq = compute_frequency_matrix(counts, pseudocount=True)
    print_matrix(pseudo_freq, "F'(b, j) - Pseudocount Frequency Matrix")

    # Log-odds matrix W(b, j)
    log_odds = compute_log_odds_matrix(pseudo_freq)
    print_matrix(log_odds, "W(b, j) - Log Odds Matrix")



 F(b, j) - Raw Frequency Matrix 
A: ['0.2963', '0.4444', '0.7778', '0.3333', '0.1481', '0.0741', '0.7778', '0.7778', '0.1111', '0.3704', '0.2963', '0.1852', '0.2593', '0.9259', '0.1481', '0.0741', '0.0741', '0.9259']
C: ['0.2593', '0.1481', '0.0370', '0.2222', '0.0741', '0.1111', '0.1111', '0.1111', '0.0370', '0.0000', '0.0741', '0.0000', '0.2593', '0.0000', '0.1111', '0.1111', '0.8889', '0.0000']
G: ['0.1111', '0.0741', '0.0370', '0.2963', '0.0741', '0.7778', '0.0741', '0.0741', '0.0000', '0.0370', '0.0000', '0.0370', '0.0000', '0.0370', '0.0000', '0.5556', '0.0000', '0.0741']
T: ['0.3333', '0.3333', '0.1481', '0.1481', '0.7037', '0.0370', '0.0370', '0.0370', '0.8519', '0.5926', '0.6296', '0.7778', '0.4815', '0.0370', '0.7407', '0.2593', '0.0370', '0.0000']

 F'(b, j) - Pseudocount Frequency Matrix 
A: ['0.2903', '0.4194', '0.7097', '0.3226', '0.1613', '0.0968', '0.7097', '0.7097', '0.1290', '0.3548', '0.2903', '0.1935', '0.2581', '0.8387', '0.1613', '0.0968', '0.0968', '0.8387']
C: 

In [ ]:
import pandas as pd

# Load the raw sequence data
sequence_data = pd.read_csv(
    r'C:\Users\kdivy\OneDrive\Desktop\COMP ANA HT DATA\ASSIGNMENT-4\E_coli_K12_MG1655.400_50.bz2',
    sep="\\", header=None, names=['gene_id', 'sequence', 'extra'], engine='python'
)
# Drop the unnecessary column
sequence_data = sequence_data.drop(columns=['extra'])

# Truncate sequences to 450 bases
sequence_data['Sequence_450bp'] = sequence_data['sequence'].str[:450]

# Rename gene_id column for clarity
sequence_data.rename(columns={'gene_id': 'Gene_ID'}, inplace=True)

# Select relevant columns
processed_gene_df = sequence_data[['Gene_ID', 'Sequence_450bp']]

# Display the resulting DataFrame
print(processed_gene_df.head())


     Gene_ID                                     Sequence_450bp
0  16127995    aacggcagaccaacatcaactgcaagctttacgcgaacgagccat...
1  16127996    ccgtttgctgcatgatattgaaaaaaatatcaccaaataaaaaac...
2  16127997    gaaacgggacgtgaactggagctggcggatattgaaattgaacct...
3  16127998    ggggattaaagtctcgacggcagaagccagggctattttaccggc...
4  16127999    aggcgaatatggcttgttcctcggcaccgcgcatccggcgaaatt...


In [ ]:
import numpy as np

# Convert the log_odds dictionary to a 2D NumPy array (rows = A/C/G/T, columns = motif positions)
PWM = np.array([log_odds[b] for b in 'acgt'])
window_size = PWM.shape[1]
nt_to_index = {'a': 0, 'c': 1, 'g': 2, 't': 3} # Create a mapping from nucleotide to row index in the PWM

# Compute top PWM score for each gene
top_PWM_scores = []
for _, row in processed_gene_df.iterrows():
    seq = row['Sequence_450bp'].lower()
    scores = [
        sum(PWM[nt_to_index[base]][i] for i, base in enumerate(seq[pos:pos+window_size]))
        for pos in range(len(seq) - window_size + 1) # Slide the motif window across the sequence
        # Only compute score if all bases are valid A/C/G/T
        if all(base in nt_to_index for base in seq[pos:pos+window_size])
    ]
    top_score = max(scores) if scores else float('-inf')
    top_PWM_scores.append((row['Gene_ID'], top_score))

# Top 30 scoring genes
top_PWM_scores_sorted = sorted(top_PWM_scores, key=lambda x: x[1], reverse=True)[:30]

# Print top 30 gene IDs and their corresponding PWM scores
for gene_id, score in top_PWM_scores_sorted:
    print(f"Gene ID: {gene_id}, PWM Score: {score:.2f}")


Gene ID: b3171 , PWM Score: 20.98
Gene ID: 16128258 , PWM Score: 20.76
Gene ID: 16132076 , PWM Score: 19.91
Gene ID: 16131126 , PWM Score: 19.09
Gene ID: 16131238 , PWM Score: 18.88
Gene ID: 16129301 , PWM Score: 18.32
Gene ID: 16128684 , PWM Score: 18.09
Gene ID: 16130583 , PWM Score: 17.88
Gene ID: 16131795 , PWM Score: 17.88
Gene ID: 16128026 , PWM Score: 17.18
Gene ID: 16129126 , PWM Score: 17.17
Gene ID: 49176132 , PWM Score: 17.03
Gene ID: 16132077 , PWM Score: 16.81
Gene ID: 16128462 , PWM Score: 16.63
Gene ID: 16131312 , PWM Score: 16.39
Gene ID: b4451 , PWM Score: 16.39
Gene ID: 16128105 , PWM Score: 16.30
Gene ID: 16131247 , PWM Score: 16.00
Gene ID: 16131392 , PWM Score: 15.93
Gene ID: 16131884 , PWM Score: 15.93
Gene ID: 16128650 , PWM Score: 15.84
Gene ID: 16129601 , PWM Score: 15.81
Gene ID: 90111311 , PWM Score: 15.81
Gene ID: 90111175 , PWM Score: 15.68
Gene ID: 16132166 , PWM Score: 15.60
Gene ID: 90111703 , PWM Score: 15.55
Gene ID: 16128795 , PWM Score: 15.16
Gene ID